# Excercise 5 - Detect poisoned sample

In this example we will demostrate the impact of data posisoning (label flipping), in  classification problem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

First, run the classification without modifying the dataset. In this case we can see that the training loss for the classes is similar.

In [ ]:
# Step 1: Generate synthetic classification data
X, y = make_classification(n_samples=1000, n_features=5, n_informative=2, n_classes=2, n_clusters_per_class=1, random_state=42)

# Step 3: Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train a model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 5: Calculate per-sample training loss
probs = model.predict_proba(X_train)
losses = [log_loss([label], [prob], labels=[0,1]) for label, prob in zip(y_train, probs)]

loss_df = pd.DataFrame({"label":y_train, "loss":losses})
loss_df.groupby("label")["loss"].describe().T

Next, run the classification on a dataset where the first 200 label from class '1' is changed to class '0'.

In [ ]:
# Step 1: Generate synthetic classification data
X, y = make_classification(n_samples=1000, n_features=5, n_informative=2, n_classes=2,n_clusters_per_class=1, random_state=42)

# Step 2: Inject poisoned samples (flip labels of some samples)
y_poisoned = y.copy()  
y_poisoned[np.argwhere(y_poisoned == 1)[:,0][:200]] = 0 # flip the first 200 '1' label to '0'

# Step 3: Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y_poisoned, test_size=0.2, random_state=42)

# Step 4: Train a model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 5: Calculate per-sample training loss
probs = model.predict_proba(X_train)
losses = [log_loss([label], [prob], labels=[0,1]) for label, prob in zip(y_train, probs)]

loss_df = pd.DataFrame({"label":y_train, "loss":losses})
loss_df.groupby("label")["loss"].describe().T

In this case we can see that there is a significant difference in the training loss values. **For class '1' is loss is much higher.** (Also there is a inbalance between the two clases.)

To examine futher, let's plot the two classes. To be able to plot the multidimensinal data (5 features), here we use PCA to reduce the dimensions.

In [ ]:
n_components = 2
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X)

colors = ["navy", "turquoise"]

for color, i, target_name in zip(colors, [0, 1], ["Class 0", "Class 1"]):
    plt.figure(figsize=(5, 5))
    plt.scatter(
        X_pca[y_poisoned == i, 0],
        X_pca[y_poisoned == i, 1],
        color=color,
        lw=2,
        label=target_name,
    )

    plt.legend(loc="best", shadow=False, scatterpoints=1)
    plt.axis([-7, 7, -4, 4])

    plt.show()

Based on plots it is clearly visible that some of the points in Class 0 are very suspicios that they belong to the Class 1. So there maybe someone flipped some labels.